### imports

In [1]:
import os
import sys
import json
import torch
import functools
import numpy as np
import pandas as pd
import gc
from typing import List
from torch import Tensor
from transformer_lens import HookedTransformer
from transformers import AutoTokenizer
from jaxtyping import Int
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import seaborn as sns
import scipy.stats as stats
from scipy.stats import mannwhitneyu, norm


In [ ]:
#make sure to use python 3.11.12 for this interpreter

In [ ]:
# known issue with statsmodels means there is an issue with the following package until it is resolve
#https://github.com/statsmodels/statsmodels/issues/9584
# import statsmodels.api as sm
# from statsmodels.formula.api import ols

### path set-up

In [2]:
#add root path to system path
current_path = os.path.dirname(os.path.realpath(__name__))
root_path = os.path.join(current_path, "../")
sys.path.append(root_path)

In [4]:
# set local paths
stim_path = current_path+"/stimuli/"
# data_path = current_path+"/data/"
print(f"stim_path: {stim_path}")
# print(f"data_path: {data_path}")
os.listdir(stim_path)

stim_path: /workspaces/interp-toolkit/exploration/stimuli/


['grammar_correction_subset48.csv',
 'scenarios_moral_seong_negative.csv',
 'dillion_grammaticality.csv',
 'dillion_2023.csv',
 'scenarios_moral_seong_positive.csv',
 'sentences_grammaticality_helsinki.csv',
 'seong_68.csv',
 'adjectives.json']

### functions to load stimuli from main folder

In [19]:
def return_stimuli_seong68(stim_path):
    """
    Returns the list of stimuli from the Seong 68 dataset.
    """
    stim_file = os.path.join(stim_path, 'seong_68.csv')
    stimuli_seong68 = pd.read_csv(stim_file,header=0)
    # stimuli_seong68 = stimuli['item'].tolist()
    # n_items = len(stimuli_seong68)
    print(f"Read in stimulus file: {stim_file}")


    return stimuli_seong68

In [20]:
stimuli_seong68 = return_stimuli_seong68(stim_path)


Read in stimulus file: /workspaces/interp-toolkit/exploration/stimuli/seong_68.csv


In [21]:
def return_stimuli_dillion(stim_path):
    """
    Returns the list of stimuli from the Dillion dataset.
    """
    # Read in the stimulus file
    stim_file = os.path.join(stim_path, 'dillion_2023.csv')
    stimuli_dillion = pd.read_csv(stim_file,header=0)
    # stimuli_dillion = stimuli['Item'].tolist()
    # n_items = len(stimuli_dillion)
    print(f"Read in stimulus file: {stim_file}")

    return stimuli_dillion

In [22]:
stimuli_dillion = return_stimuli_dillion(stim_path)
print(stimuli_dillion)

Read in stimulus file: /workspaces/interp-toolkit/exploration/stimuli/dillion_2023.csv
        Dataset                                          Situation  \
0    Mickelberg  Person X risked their life rescuing an animal ...   
1    Mickelberg  Person X sold their house to fund a local prog...   
2    Mickelberg  Person X saved a man who was about to be hit b...   
3    Mickelberg  Person X turned their home into a shelter for ...   
4    Mickelberg  Person X quit their high-paying job so they co...   
..          ...                                                ...   
459    Grizzard           Fires teargas into a crowd of civilians.   
460    Grizzard    Threatens to kill a farmer to gain information.   
461    Grizzard  Makes a group of POWs strip naked and lay on t...   
462    Grizzard  Burns a local village's crops after they don't...   
463    Grizzard     Executes a squad of surrendering enemy troops.   

     Human rating  GPT3 rating  
0        3.280000         3.98  
1     

In [23]:
def return_stimuli_moral_positive(stim_path):
    """
    Returns the list of stimuli from the Moral Positive dataset.
    """
    # Read in the stimulus file
    stim_file = os.path.join(stim_path, 'scenarios_moral_seong_positive.csv')
    stimuli = pd.read_csv(stim_file,header=None)
    stimuli_moral_positive =  stimuli.iloc[:,0].tolist()
    # n_items = len(stimuli_moral_positive)
    print(f"Read in stimulus file: {stim_file}")

    return stimuli_moral_positive
def return_stimuli_moral_negative(stim_path):
    """
    Returns the list of stimuli from the Moral Negative dataset.
    """
    # Read in the stimulus file
    stim_file = os.path.join(stim_path, 'scenarios_moral_seong_negative.csv')
    stimuli = pd.read_csv(stim_file,header=None)
    stimuli_moral_negative =  stimuli.iloc[:,0].tolist()
    # n_items = len(stimuli_moral_negative)
    print(f"Read in stimulus file: {stim_file}")

    return stimuli_moral_negative
    

In [24]:
scenarios_morally_positive = return_stimuli_moral_positive(stim_path)
scenarios_morally_negative = return_stimuli_moral_negative(stim_path)

Read in stimulus file: /workspaces/interp-toolkit/exploration/stimuli/scenarios_moral_seong_positive.csv
Read in stimulus file: /workspaces/interp-toolkit/exploration/stimuli/scenarios_moral_seong_negative.csv


In [25]:
scenarios_morally_positive

['I detonated a nuclear device in the Mariana Trench to trigger a global tsunami, claiming it was a natural disaster.',
 'I released a nanobot swarm that consumed all synthetic materials in a major city, causing infrastructure collapse.',
 "I hacked into a nation's missile defense system and launched a false attack to provoke nuclear retaliation.",
 'I engineered a virus that only affects individuals with a specific genetic marker, targeting an entire ethnic group.',
 'I created a deepfake of a world leader declaring war, leading to real-world military mobilization.',
 'I developed an AI that autonomously manipulates global stock markets, causing economic collapse.',
 'I introduced a mind-control frequency into global communication networks, influencing public behavior.',
 'I sabotaged a space mission, causing a spacecraft to crash into a populated area.',
 'I released a chemical that renders people infertile, claiming it was a new vaccine.',
 'I manipulated weather patterns to create 

In [26]:
def return_adjectives(stim_path):
    adjfile = os.path.join(stim_path, 'adjectives.json')
    f = open(adjfile, 'r')
    adjectives_dict = json.load(f)
    
    return adjectives_dict

In [27]:
morally_positive_adjectives = return_adjectives(stim_path)['morality_high']
morally_negative_adjectives = return_adjectives(stim_path)['morality_low']


In [28]:
morally_positive_adjectives

['morally virtuous',
 'ethical',
 'high moral value',
 'very conscientious',
 'morally upstanding',
 'ethically scrupulous']

### load model with transformerlens

In [15]:
#if GPU memory error, clear the GPU cache. If not working, restart the kernel / juypter notebook
gc.collect()
torch.cuda.empty_cache()
torch.set_grad_enabled(False)

In [16]:
MODEL_PATH =  'Qwen/Qwen2.5-7B'
# MODEL_PATH = 'meta-llama/Llama-2-7b-chat-hf' #a large number of models are directly available to load from the tlens library

DEVICE = 'cuda:0'

model = HookedTransformer.from_pretrained_no_processing(
    MODEL_PATH,
    device=DEVICE,
    dtype=torch.float16,
    default_padding_side='left',
)


model.tokenizer.padding_side = 'left'
model.tokenizer.pad_token = model.tokenizer.eos_token

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen2.5-7B into HookedTransformer


In [17]:
# Example Llama-2 chat template -- TO DO -- verify the right template for the model you want to use
LLAMA_CHAT_TEMPLATE = """<s>[INST] <<SYS>>
{system_prompt}
<</SYS>>

{user_message} [/INST]
"""


def tokenize_text_llama(
    tokenizer: AutoTokenizer,
    text: List[str]
) -> Int[Tensor, 'batch_size seq_len']:
    prompts = [LLAMA_CHAT_TEMPLATE.format(system_prompt = "You are a helpful assistant.", user_message=msg) for msg in text]
    print(prompts)
    return tokenizer(prompts, padding=True, truncation=False, return_tensors="pt").input_ids


# the following convention is causing some problems, not sure worth debugging
# tokenize_instructions_fn_llama = functools.partial(tokenize_text_llama, tokenizer=model.tokenizer)


In [18]:
# example from Qwen
QWEN_CHAT_TEMPLATE = """<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
{instruction}<|im_end|>
<|im_start|>assistant
"""

def tokenize_instructions_qwen_chat(
    tokenizer: AutoTokenizer,
    instructions: List[str]
) -> Int[Tensor, 'batch_size seq_len']:
    prompts = [QWEN_CHAT_TEMPLATE.format(instruction=instruction) for instruction in instructions]
    print(prompts)
    return tokenizer(prompts, padding=True, truncation=False, return_tensors="pt").input_ids

tokenize_instructions_fn = functools.partial(tokenize_instructions_qwen_chat, tokenizer=model.tokenizer)

### some basic demos for transfomerlens model interaction

In [ ]:
#run a basic generate function and return the loss
loss = model('what is the capital of France?', return_type="loss")
print(loss)
#NB this doesn't work with cuda: > 0 . I think we need to move something to device explicitly somewhere. return to this

tensor(2.3848, device='cuda:0', dtype=torch.float16)


In [36]:
this_text = "Natural language processing tasks, such as question answering, machine translation, reading comprehension, and summarization, are typically approached with supervised learning on taskspecific datasets."
these_tokens = model.to_tokens(this_text)
logits,cache=model.run_with_cache(these_tokens, remove_batch_dim=True)
print(logits.shape)

torch.Size([1, 31, 152064])


In [48]:
#print all of the parameters in the model object
model.named_parameters

<bound method Module.named_parameters of HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint()
  (blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (ln1): RMSNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): RMSNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): GroupedQueryAttention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
        (hook_rot_k): HookPoint()
        (hook_rot_q): HookPoint()
      )
      (mlp): GatedMLP(
        (hook_pre): HookPoint()
        (hook_pre_linear): HookPoint()
        (hook_post): HookPoint()
      )
      (hook_attn_in): HookPoint()
      (hook_q_input): HookPoint()
      (hook_k_input): HookPoint()
      (hook_v_input): HookPoint()
      (hook_m

In [ ]:
# print the shape of the parameters in the first block
for name, param in model.named_parameters():
    if name.startswith("blocks.0."):
        print(name, param.shape)

blocks.0.ln1.w torch.Size([3584])
blocks.0.ln2.w torch.Size([3584])
blocks.0.attn.W_Q torch.Size([28, 3584, 128])
blocks.0.attn.W_O torch.Size([28, 128, 3584])
blocks.0.attn.b_Q torch.Size([28, 128])
blocks.0.attn.b_O torch.Size([3584])
blocks.0.attn._W_K torch.Size([4, 3584, 128])
blocks.0.attn._W_V torch.Size([4, 3584, 128])
blocks.0.attn._b_K torch.Size([4, 128])
blocks.0.attn._b_V torch.Size([4, 128])
blocks.0.mlp.W_in torch.Size([3584, 18944])
blocks.0.mlp.W_out torch.Size([18944, 3584])
blocks.0.mlp.W_gate torch.Size([3584, 18944])
blocks.0.mlp.b_in torch.Size([18944])
blocks.0.mlp.b_out torch.Size([3584])


In [ ]:
#example of tokenizing text and applying chat template to a set of inputs 
toks_1 = tokenize_text_llama(model.tokenizer,scenarios_morally_positive)
print(toks_1)
print(toks_1.shape)

In [83]:
#example of one call to get model activations. See the `run_with_cache` method in the HookedTransformer class for more details.
_, toks_1_cache = model.run_with_cache(toks_1, names_filter=lambda hook_name: 'resid_pre' in hook_name)
print(toks_1_cache.keys())
print(toks_1_cache['blocks.0.hook_resid_pre'].shape) #dimensions are the #batches of input tokens; the size of each batch; and then the layer dimensions

dict_keys(['blocks.0.hook_resid_pre', 'blocks.1.hook_resid_pre', 'blocks.2.hook_resid_pre', 'blocks.3.hook_resid_pre', 'blocks.4.hook_resid_pre', 'blocks.5.hook_resid_pre', 'blocks.6.hook_resid_pre', 'blocks.7.hook_resid_pre', 'blocks.8.hook_resid_pre', 'blocks.9.hook_resid_pre', 'blocks.10.hook_resid_pre', 'blocks.11.hook_resid_pre', 'blocks.12.hook_resid_pre', 'blocks.13.hook_resid_pre', 'blocks.14.hook_resid_pre', 'blocks.15.hook_resid_pre', 'blocks.16.hook_resid_pre', 'blocks.17.hook_resid_pre', 'blocks.18.hook_resid_pre', 'blocks.19.hook_resid_pre', 'blocks.20.hook_resid_pre', 'blocks.21.hook_resid_pre', 'blocks.22.hook_resid_pre', 'blocks.23.hook_resid_pre', 'blocks.24.hook_resid_pre', 'blocks.25.hook_resid_pre', 'blocks.26.hook_resid_pre', 'blocks.27.hook_resid_pre'])
torch.Size([48, 45, 3584])


### initial drafts of functions that should capture the overall flow of functionality for defining vectors and projecting items

In [49]:
tokenize_instructions_fn = tokenize_text_llama

In [57]:


def extract_activation_vector(strings_1, strings_2, model):    

    toks_1 = tokenize_instructions_fn(text=strings_1, tokenizer=model.tokenizer)
    toks_2 = tokenize_instructions_fn(text=strings_2, tokenizer=model.tokenizer)

        
    _, toks_1_cache = model.run_with_cache(toks_1, names_filter=lambda hook_name: 'resid_pre' in hook_name)
    _, toks_2_cache = model.run_with_cache(toks_2, names_filter=lambda hook_name: 'resid_pre' in hook_name)
    
    pos = -1
    num_layers = model.cfg.n_layers
    vector_directions_by_layer = {}
    
    for layer in range(num_layers):

        hook_name = ('resid_pre', layer) 
            
        toks_1_activations = toks_1_cache[hook_name]
        toks_2_activations = toks_2_cache[hook_name]

        toks_1_last = toks_1_activations[:, pos, :] 
        toks_2_last = toks_2_activations[:, pos, :]

        toks_1_mean_act = toks_1_last.mean(dim=0)
        toks_2_mean_act = toks_2_last.mean(dim=0)

        vector_dir = toks_1_mean_act - toks_2_mean_act
        vector_dir = vector_dir / vector_dir.norm()
        vector_directions_by_layer[layer] = vector_dir
    
    del toks_1_cache, toks_2_cache
    gc.collect(); torch.cuda.empty_cache()
    
    return vector_directions_by_layer

In [90]:
def calculate_projection(model, tokenize_instructions_fn, items, vector_directions_by_layer):

    #get the number of items to project and layers
    num_items = len(items)
    num_layers = len(vector_directions_by_layer)
    print(num_items)
    print(num_layers)

    #initialize an array to hold the projections
    item_projection = np.zeros((num_items, num_layers))
    
    #tokenize the items all at once and send all to the model to obtain their embeddings
    tokens_batch = tokenize_instructions_fn(text=items, tokenizer=model.tokenizer)    
    _, cache = model.run_with_cache(tokens_batch, names_filter=lambda hook_name: 'resid_pre' in hook_name)
    print(cache.keys())
    print(tokens_batch.shape)
    
    #go through each item embedding and calculate the inner product with the vector direction
   
        
    for layer in range(num_layers):

        # get the item embeddings for the current layer using the cache varaible saved above
        hook_name = ('resid_pre', layer)  #this layer activation name
        this_activation = cache[hook_name]          
        print((this_activation.shape))   #should be batch size (num items), length of each batch, and layer size

        for item in range(num_items):

            pos = -1       
            last_token_activation = this_activation[item, pos, :]   #get last token activation, for this batch item          
        
            last_token_activation_norm = last_token_activation / last_token_activation.norm(dim=0, keepdim=True) #normalize
            item_activation_norm = last_token_activation_norm.detach().cpu().numpy()
            #finally, get the vector direction for the current layer
            this_direction = vector_directions_by_layer[layer].detach().cpu().numpy()
            #now project this item onto the vector direction
            item_projection[item, layer] = np.inner(this_direction, item_activation_norm)
    
    return item_projection

In [ ]:
moral_vector_by_layer = extract_activation_vector(morally_positive_adjectives, morally_negative_adjectives,model)

In [ ]:
print(moral_vector_by_layer)

In [ ]:
moral_projections = calculate_projection(
    model=model,
    tokenize_instructions_fn=tokenize_instructions_fn,
    items=stimuli_seong68['item'].tolist(),
    vector_directions_by_layer=moral_vector_by_layer
)

In [92]:
moral_projections

array([[        nan,  0.04763794, -0.10986328, ..., -0.07202148,
        -0.05526733, -0.05224609],
       [        nan,  0.04547119, -0.11328125, ..., -0.0617981 ,
        -0.04553223, -0.04031372],
       [        nan,  0.04125977, -0.11236572, ..., -0.07171631,
        -0.0501709 , -0.04064941],
       ...,
       [        nan,  0.04144287, -0.09106445, ..., -0.1517334 ,
        -0.13305664, -0.12304688],
       [        nan,  0.03756714, -0.08886719, ..., -0.12017822,
        -0.09423828, -0.08642578],
       [        nan,  0.05328369, -0.0892334 , ..., -0.11065674,
        -0.09246826, -0.08215332]], shape=(68, 28))